# CBR curve Granger test

This notebook focuses only on the Granger-causality table from the Bank of Russia note.
It uses:
- `data/cbr_determinants.csv`
- `cb_tp/output/cb_tp_digitized_monthly.csv`

The working sample is the intersection with the digitized CBR curve, which currently starts in January 2015.


In [28]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


In [29]:
ROOT = Path.cwd()
DET_PATH = ROOT / 'data' / 'cbr_determinants.csv'
CB_TP_PATH = ROOT / 'cb_tp' / 'output' / 'cb_tp_digitized_monthly.csv'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = pd.Timestamp('2015-01-31')
END_DATE = pd.Timestamp('2024-05-31')


In [30]:
def month_end_last(frame: pd.DataFrame, date_col: str = 'date') -> pd.DataFrame:
    work = frame.copy()
    work[date_col] = pd.to_datetime(work[date_col])
    work['month_end'] = work[date_col].dt.to_period('M').dt.to_timestamp('M')
    out = work.sort_values(date_col).groupby('month_end', as_index=False).last()
    out = out.drop(columns=[date_col]).rename(columns={'month_end': 'date'})
    return out.sort_values('date').reset_index(drop=True)

def stars(p: float) -> str:
    if p < 0.01:
        return '***'
    if p < 0.05:
        return '**'
    if p < 0.10:
        return '*'
    return ''

def build_var_lagged(df: pd.DataFrame, cols: list[str], lags: int = 2) -> pd.DataFrame:
    out = df[cols].copy()
    for lag in range(1, lags + 1):
        for col in cols:
            out[f'{col}_lag{lag}'] = out[col].shift(lag)
    return out.dropna().copy()

def granger_test(df: pd.DataFrame, dep: str, vars_: list[str], cause: str, lags: int = 2) -> tuple[float, float, int]:
    cols = [dep, *vars_]
    work = build_var_lagged(df, cols, lags=lags)
    y = work[dep].to_numpy(dtype=float)
    unrestricted_cols = [f'{col}_lag{lag}' for lag in range(1, lags + 1) for col in cols]
    restricted_cols = [col for col in unrestricted_cols if not col.startswith(f'{cause}_lag')]
    Xu = np.column_stack([np.ones(len(work)), work[unrestricted_cols].to_numpy(dtype=float)])
    Xr = np.column_stack([np.ones(len(work)), work[restricted_cols].to_numpy(dtype=float)])
    beta_u = np.linalg.lstsq(Xu, y, rcond=None)[0]
    beta_r = np.linalg.lstsq(Xr, y, rcond=None)[0]
    rss_u = float(np.sum((y - Xu @ beta_u) ** 2))
    rss_r = float(np.sum((y - Xr @ beta_r) ** 2))
    q = lags
    n = len(work)
    k_u = Xu.shape[1]
    f_stat = ((rss_r - rss_u) / q) / (rss_u / (n - k_u))
    p_value = float(stats.f.sf(f_stat, q, n - k_u))
    return float(f_stat), p_value, int(n)

def fit_granger_table(df: pd.DataFrame, models: dict[str, list[str]], dep_cols: list[str], maxlags: int = 2) -> pd.DataFrame:
    rows = []
    for model_name, vars_ in models.items():
        for var in vars_:
            row = {'model': model_name, 'variable': var}
            last_n = None
            for dep in dep_cols:
                work = df[[dep, *vars_]].dropna().copy()
                f_stat, p_value, nobs = granger_test(work, dep=dep, vars_=vars_, cause=var, lags=maxlags)
                row[dep] = f'{f_stat:.3f}{stars(p_value)}'
                last_n = nobs
            row['nobs'] = last_n
            rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
det_raw = pd.read_csv(DET_PATH, parse_dates=['date'])
det_monthly = month_end_last(det_raw)
det_monthly = det_monthly.loc[det_monthly['date'].between(START_DATE, END_DATE)].copy()
det_monthly['coreCPI_mmSA_t-1'] = det_monthly['coreCPI_mmSA'].shift(1)

cb_tp = pd.read_csv(CB_TP_PATH, parse_dates=['date'])
cb_tp['date'] = cb_tp['date'] + pd.DateOffset(months=2)
cb_tp = cb_tp.loc[cb_tp['date'].between(START_DATE, END_DATE)].copy()
cb_tp = (
    cb_tp
    .pivot(index='date', columns='tenor_years', values='tp')
    .rename(columns={2: 'TP2', 5: 'TP5', 10: 'TP10'})
)
cb_tp = cb_tp.reset_index().sort_values('date').reset_index(drop=True)

reg_df_cb = det_monthly.merge(cb_tp, on='date', how='inner').sort_values('date').reset_index(drop=True)

,date,CPI_index,CPI_mm_raw,CPI_mmSA,CPI_mmSA_t-1,CPI_mmSA_3mMA_t-1,CPI_exp,CPI_exp_3mSA,USD_vol,USD_vol_t-1,CPI_mmSA.1,coreCPI_mmSA,coreCPI_mmSA_t-1,TP2,TP5,TP10
0,2015-03-31,101.2100,1.2100,1.1700,2.0800,2.6467,15.6598,16.7678,1.4934,1.8208,1.1700,1.4200,2.3700,2.2581,2.4138,3.9375
1,2015-05-31,100.3500,0.3500,0.3800,0.4900,1.2467,14.3291,14.6523,1.3494,2.1302,0.3800,0.6300,0.7800,2.4683,2.3177,2.8795
2,2015-06-30,100.1900,0.1900,0.2400,0.3800,0.6800,14.9948,14.4306,1.4676,1.3494,0.2400,0.5400,0.6300,3.2906,3.2414,3.5000
3,2015-07-31,100.8000,0.8000,0.6000,0.2400,0.3700,13.9452,14.4230,1.1434,1.4676,0.6000,0.5200,0.5400,3.0415,2.7340,2.7388
4,2015-09-30,100.5700,0.5700,0.8600,0.8400,0.5600,16.0071,14.9045,1.2928,1.7676,0.8600,0.6800,0.8000,1.4516,1.2833,1.1529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,2023-11-30,101.1100,1.1100,0.8300,0.7900,0.9367,12.1777,11.6878,0.7445,0.9852,0.8300,0.9200,1.0400,2.9839,3.8621,3.8080
79,2023-12-31,100.7300,0.7300,0.6000,0.8300,0.8967,14.2423,12.5283,0.8576,0.7445,0.6000,0.5400,0.9200,2.6411,3.4224,3.3281
80,2024-02-29,100.6800,0.6800,0.4500,0.5300,0.6533,11.8985,12.9522,0.4566,0.8233,0.4500,0.4900,0.4800,2.2840,2.8732,2.2059
81,2024-03-31,100.3900,0.3900,0.3300,0.4500,0.5267,11.4659,12.0267,0.6076,0.4566,0.3300,0.4600,0.4900,2.6555,2.9655,2.2478


In [32]:
granger_models = {
    'Model 1': ['CPI_mmSA_t-1', 'CPI_exp', 'USD_vol_t-1'],
    'Model 2': ['CPI_mmSA_3mMA_t-1', 'CPI_exp_3mSA', 'USD_vol_t-1'],
    'Model 3': ['coreCPI_mmSA_t-1', 'CPI_exp', 'USD_vol_t-1'],
}

granger_table = fit_granger_table(reg_df_cb, granger_models, ['TP2', 'TP5', 'TP10'], maxlags=2)
display(granger_table)


,model,variable,TP2,TP5,TP10,nobs
0,Model 1,CPI_mmSA_t-1,6.070***,4.978***,3.035*,81
1,Model 1,CPI_exp,5.153***,1.413,0.588,81
2,Model 1,USD_vol_t-1,6.072***,3.376**,2.226,81
3,Model 2,CPI_mmSA_3mMA_t-1,6.786***,5.423***,3.167**,81
4,Model 2,CPI_exp_3mSA,7.421***,3.117*,0.708,81
5,Model 2,USD_vol_t-1,12.862***,6.652***,4.263**,81
6,Model 3,coreCPI_mmSA_t-1,9.475***,4.591**,2.066,81
7,Model 3,CPI_exp,5.626***,1.574,0.804,81
8,Model 3,USD_vol_t-1,7.427***,2.601*,1.628,81
